# New York City Airbnb Data -- Cleaning, Exploration, and Visual Analysis

**Project:** Comprehensive Data Story on NYC Airbnb Open Data (2019)  
**Objective:** Understand pricing dynamics, geographic distribution, host behaviour, and guest preferences across New York City boroughs to support data-driven decision-making for hosts, guests, and urban policy stakeholders.  
**Dataset:** [Kaggle -- New York City Airbnb Open Data](https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data)  

---

## 1. Setup and Library Imports

We begin by importing the standard data science stack. Pandas handles the tabular data, NumPy provides numerical operations, and Matplotlib together with Seaborn gives us publication-quality static plots. Plotly is used later for interactive visualisations. WordCloud helps surface common listing themes from free-text names.

In [ ]:
# --- Standard Libraries ---
import pandas as pd
import numpy as np
import warnings

# --- Visualization Libraries ---
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Text Analysis ---
from wordcloud import WordCloud

# --- Configuration ---
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print('All libraries loaded successfully.')

## 2. Loading the Dataset

The raw CSV sits in the `archive/` directory. We load it and immediately take a first look at the shape, column types, and a handful of rows to form an initial mental model of the data.

In [ ]:
# Load the dataset from the archive folder
raw_df = pd.read_csv('../archive/AB_NYC_2019.csv')

print(f'Dataset shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns')
print()
raw_df.head(10)

In [ ]:
# Quick overview of column types and memory usage
raw_df.info()

In [ ]:
# Basic descriptive statistics for numerical columns
raw_df.describe()

### Initial Observations

A few things stand out right away:

- The **price** column has a maximum of $10,000 per night, which very likely includes erroneous or extreme luxury listings that could distort averages.
- **minimum_nights** reaches 1,250 days, suggesting some listings are not genuine short-term rentals.
- Several columns have missing values that we will need to handle carefully.

---

## 3. Data Quality Assessment

Before we can draw any conclusions, we need to understand and address data quality issues. This section systematically examines missing values, duplicates, and data type inconsistencies.

In [ ]:
# --- Missing Value Analysis ---
missing_counts = raw_df.isnull().sum()
missing_pct = (raw_df.isnull().sum() / len(raw_df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing Percentage': missing_pct
})

# Show only columns that actually have missing data
missing_summary = missing_summary[missing_summary['Missing Count'] > 0]
missing_summary = missing_summary.sort_values('Missing Percentage', ascending=False)

print('Columns with missing values:')
print()
missing_summary

In [ ]:
# Visualise the missing data pattern
fig, ax = plt.subplots(figsize=(10, 4))

colours = ['#2ecc71' if pct == 0 else '#e74c3c' for pct in (raw_df.isnull().sum() / len(raw_df) * 100)]
bars = ax.barh(
    raw_df.columns,
    raw_df.isnull().sum() / len(raw_df) * 100,
    color=colours,
    edgecolor='white',
    linewidth=0.5
)

ax.set_xlabel('Missing Values (%)')
ax.set_title('Missing Data Profile Across All Columns', fontweight='bold')
ax.invert_yaxis()

# Annotate bars with actual percentages where missing data exists
for bar_item, pct in zip(bars, raw_df.isnull().sum() / len(raw_df) * 100):
    if pct > 0:
        ax.text(bar_item.get_width() + 0.3, bar_item.get_y() + bar_item.get_height()/2,
                f'{pct:.1f}%', va='center', fontsize=9, color='#e74c3c')

plt.tight_layout()
plt.savefig('missing_data_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print('Missing data profile saved.')

In [ ]:
# --- Duplicate Check ---
duplicate_count = raw_df.duplicated().sum()
duplicate_id_count = raw_df['id'].duplicated().sum()

print(f'Exact duplicate rows:     {duplicate_count}')
print(f'Duplicate listing IDs:    {duplicate_id_count}')
print()

if duplicate_count == 0 and duplicate_id_count == 0:
    print('No duplicates detected. Each row represents a unique listing.')

## 4. Data Cleaning

With the quality issues mapped out, we now apply a series of cleaning steps. The guiding principle is to be conservative: we fill what we can reasonably infer, drop what is truly unusable, and document every decision.

In [ ]:
# Work on a copy so we can always go back to the raw data if needed
df = raw_df.copy()

# ---- Step 1: Handle missing values ----

# 'name' column -- a small number of listings have no name.
# These are not critical for analysis, so we fill with a placeholder.
df['name'] = df['name'].fillna('Unnamed Listing')

# 'host_name' -- similarly, a few hosts have no recorded name.
df['host_name'] = df['host_name'].fillna('Unknown Host')

# 'last_review' and 'reviews_per_month' -- these are missing for listings
# that have never been reviewed (number_of_reviews == 0). This is expected.
# We fill reviews_per_month with 0 for these cases.
df['reviews_per_month'] = df['reviews_per_month'].fillna(0)

# Convert last_review to datetime for proper temporal analysis
df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')

print('Step 1 complete: Missing values handled.')
print(f'  Remaining missing values: {df.isnull().sum().sum()}')

In [ ]:
# ---- Step 2: Handle extreme values and outliers ----

# Listings with price = 0 are not meaningful (likely errors or placeholders)
zero_price_count = (df['price'] == 0).sum()
print(f'Listings with price = $0: {zero_price_count}')
df = df[df['price'] > 0]

# Listings with minimum_nights > 365 are effectively long-term leases,
# not short-term rentals. We exclude them to keep the analysis relevant.
long_stay_count = (df['minimum_nights'] > 365).sum()
print(f'Listings with minimum_nights > 365: {long_stay_count}')
df = df[df['minimum_nights'] <= 365]

print(f'\nDataset after removing extreme values: {df.shape[0]:,} rows')
print(f'Rows removed: {raw_df.shape[0] - df.shape[0]:,}')

In [ ]:
# ---- Step 3: Feature engineering ----

# Create a price category for easier grouping in visualisations
price_bins = [0, 50, 100, 200, 500, 10001]
price_labels = ['Budget ($0-50)', 'Economy ($51-100)', 'Mid-Range ($101-200)',
                'Premium ($201-500)', 'Luxury ($500+)']
df['price_category'] = pd.cut(df['price'], bins=price_bins, labels=price_labels, right=True)

# Extract month and year from last_review for temporal analysis
df['review_month'] = df['last_review'].dt.month
df['review_year'] = df['last_review'].dt.year

# Estimated annual revenue: a rough proxy using reviews.
# Airbnb's own research suggests roughly 1 review per 2 stays on average.
# So estimated_stays = number_of_reviews * 2, and revenue = stays * price * avg_stay_length.
# We use a conservative 3-night average stay.
df['estimated_annual_revenue'] = df['reviews_per_month'] * 12 * 2 * df['price'] * 3

# Host category based on number of listings
df['host_type'] = df['calculated_host_listings_count'].apply(
    lambda x: 'Individual Host (1 listing)' if x == 1
    else 'Small Portfolio (2-5)' if x <= 5
    else 'Professional Host (6+)'
)

print('Step 3 complete: New features created.')
print(f'  New columns: price_category, review_month, review_year, estimated_annual_revenue, host_type')
print()
df.head()

In [ ]:
# ---- Final cleaned dataset summary ----
print('=== Cleaned Dataset Summary ===')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Missing values remaining: {df.isnull().sum().sum()}')
print()
df.describe()

---

## 5. Exploratory Data Analysis

Now that the data is clean and enriched, we explore it through a series of focused questions. Each subsection addresses a specific angle that matters for our data story.

### 5.1 Geographic Distribution -- Where are the listings?

In [ ]:
# Borough-level listing counts
borough_counts = df['neighbourhood_group'].value_counts().reset_index()
borough_counts.columns = ['Borough', 'Listing Count']
borough_counts['Percentage'] = (borough_counts['Listing Count'] / borough_counts['Listing Count'].sum() * 100).round(1)

print('Listings by Borough:')
borough_counts

In [ ]:
# Borough distribution -- side-by-side bar and pie chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

borough_palette = {'Manhattan': '#3498db', 'Brooklyn': '#e74c3c', 'Queens': '#2ecc71',
                   'Bronx': '#f39c12', 'Staten Island': '#9b59b6'}
borough_order = ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']

# Bar chart
sns.barplot(data=borough_counts, x='Borough', y='Listing Count',
            palette=borough_palette, order=borough_order, ax=axes[0], edgecolor='white')
axes[0].set_title('Number of Airbnb Listings by Borough', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Count')
axes[0].set_xlabel('')

# Add count labels on bars
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=10, fontweight='bold')

# Pie chart
colours_pie = [borough_palette[b] for b in borough_order]
axes[1].pie(borough_counts.set_index('Borough').loc[borough_order, 'Listing Count'],
            labels=borough_order, autopct='%1.1f%%', colors=colours_pie,
            startangle=140, textprops={'fontsize': 11})
axes[1].set_title('Market Share by Borough', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.savefig('borough_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top 20 neighbourhoods by listing count
top_neighbourhoods = df['neighbourhood'].value_counts().head(20).reset_index()
top_neighbourhoods.columns = ['Neighbourhood', 'Count']

# Map each neighbourhood to its borough for colour coding
neighbourhood_to_borough = df[['neighbourhood', 'neighbourhood_group']].drop_duplicates()
top_neighbourhoods = top_neighbourhoods.merge(
    neighbourhood_to_borough, left_on='Neighbourhood', right_on='neighbourhood', how='left'
)

fig, ax = plt.subplots(figsize=(12, 8))
palette_mapped = [borough_palette.get(b, '#999') for b in top_neighbourhoods['neighbourhood_group']]

bars = ax.barh(top_neighbourhoods['Neighbourhood'][::-1], top_neighbourhoods['Count'][::-1],
               color=palette_mapped[::-1], edgecolor='white', linewidth=0.5)
ax.set_xlabel('Number of Listings')
ax.set_title('Top 20 Neighbourhoods by Listing Count', fontweight='bold', fontsize=14)

# Add a simple legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=borough_palette[b], label=b) for b in borough_order]
ax.legend(handles=legend_elements, title='Borough', loc='lower right')

plt.tight_layout()
plt.savefig('top_neighbourhoods.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Geographic scatter plot -- every listing on the NYC map
fig = px.scatter_mapbox(
    df,
    lat='latitude',
    lon='longitude',
    color='neighbourhood_group',
    color_discrete_map=borough_palette,
    size='price',
    size_max=8,
    opacity=0.4,
    hover_name='name',
    hover_data={'price': ':$,.0f', 'room_type': True, 'neighbourhood': True,
                'latitude': False, 'longitude': False, 'neighbourhood_group': False},
    title='Geographic Distribution of Airbnb Listings Across NYC',
    labels={'neighbourhood_group': 'Borough'},
    mapbox_style='carto-positron',
    zoom=10,
    center={'lat': 40.7128, 'lon': -74.0060},
    width=900,
    height=650
)

fig.update_layout(
    title_font_size=16,
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.show()

**Key Finding:** Manhattan and Brooklyn together account for nearly 85% of all Airbnb listings in NYC. Williamsburg, Bedford-Stuyvesant, and Harlem emerge as the three densest neighbourhoods. The outer boroughs (Bronx, Staten Island) are significantly underrepresented, presenting potential growth opportunities or reflecting genuine demand asymmetries.

---

### 5.2 Pricing Analysis -- What does it cost to stay?

In [ ]:
# Overall price distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram -- clipped at $500 for readability (covers 96%+ of listings)
df_price_clipped = df[df['price'] <= 500]

axes[0].hist(df_price_clipped['price'], bins=80, color='#3498db', edgecolor='white',
             linewidth=0.3, alpha=0.85)
axes[0].axvline(df['price'].median(), color='#e74c3c', linestyle='--', linewidth=2,
                label=f'Median: ${df["price"].median():.0f}')
axes[0].axvline(df['price'].mean(), color='#f39c12', linestyle='--', linewidth=2,
                label=f'Mean: ${df["price"].mean():.0f}')
axes[0].set_xlabel('Price per Night ($)')
axes[0].set_ylabel('Number of Listings')
axes[0].set_title('Price Distribution (Up to $500)', fontweight='bold', fontsize=13)
axes[0].legend(fontsize=10)

# Box plot by borough
sns.boxplot(data=df[df['price'] <= 500], x='neighbourhood_group', y='price',
            order=borough_order, palette=borough_palette, ax=axes[1],
            fliersize=1, linewidth=1)
axes[1].set_xlabel('')
axes[1].set_ylabel('Price per Night ($)')
axes[1].set_title('Price Distribution by Borough', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.savefig('price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Overall median price: ${df["price"].median():.0f}')
print(f'Overall mean price:   ${df["price"].mean():.0f}')

In [ ]:
# Detailed borough-level price statistics
price_by_borough = df.groupby('neighbourhood_group')['price'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
).round(2)
price_by_borough.columns = ['Listing Count', 'Mean Price', 'Median Price',
                            'Std Dev', 'Min Price', 'Max Price']
price_by_borough = price_by_borough.sort_values('Median Price', ascending=False)

print('Price Statistics by Borough:')
price_by_borough

In [ ]:
# Price by room type -- violin plot for richer distributional insight
fig, ax = plt.subplots(figsize=(12, 6))

room_palette = {'Entire home/apt': '#3498db', 'Private room': '#2ecc71', 'Shared room': '#f39c12'}
room_order = ['Entire home/apt', 'Private room', 'Shared room']

sns.violinplot(data=df[df['price'] <= 500], x='room_type', y='price',
               order=room_order, palette=room_palette, ax=ax,
               inner='quartile', linewidth=1, cut=0)

ax.set_xlabel('Room Type', fontsize=12)
ax.set_ylabel('Price per Night ($)', fontsize=12)
ax.set_title('Price Distribution by Room Type', fontweight='bold', fontsize=14)

# Annotate medians
for i, rt in enumerate(room_order):
    median_val = df[df['room_type'] == rt]['price'].median()
    ax.text(i, median_val + 10, f'Median: ${median_val:.0f}',
            ha='center', fontsize=10, fontweight='bold', color='#2c3e50')

plt.tight_layout()
plt.savefig('price_by_room_type.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: Average price by borough and room type
price_heatmap_data = df.groupby(['neighbourhood_group', 'room_type'])['price'].median().unstack()
price_heatmap_data = price_heatmap_data.reindex(borough_order)
price_heatmap_data = price_heatmap_data[room_order]

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(price_heatmap_data, annot=True, fmt=',.0f', cmap='YlOrRd',
            linewidths=1, linecolor='white', cbar_kws={'label': 'Median Price ($)'},
            ax=ax)
ax.set_title('Median Nightly Price ($) -- Borough vs Room Type', fontweight='bold', fontsize=13)
ax.set_xlabel('Room Type')
ax.set_ylabel('Borough')

plt.tight_layout()
plt.savefig('price_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Price category breakdown
price_cat_counts = df['price_category'].value_counts().reset_index()
price_cat_counts.columns = ['Category', 'Count']
price_cat_counts['Percentage'] = (price_cat_counts['Count'] / price_cat_counts['Count'].sum() * 100).round(1)
price_cat_counts = price_cat_counts.sort_values('Category')

fig, ax = plt.subplots(figsize=(10, 5))
category_colors = ['#27ae60', '#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
bars = ax.bar(price_cat_counts['Category'], price_cat_counts['Count'],
              color=category_colors, edgecolor='white', linewidth=1)

for bar_item, pct in zip(bars, price_cat_counts['Percentage']):
    ax.text(bar_item.get_x() + bar_item.get_width()/2, bar_item.get_height() + 100,
            f'{pct:.1f}%', ha='center', fontweight='bold', fontsize=10)

ax.set_ylabel('Number of Listings')
ax.set_title('Listings by Price Category', fontweight='bold', fontsize=14)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('price_categories.png', dpi=150, bbox_inches='tight')
plt.show()

**Key Finding:** The price distribution is heavily right-skewed, with the median ($106) significantly lower than the mean ($152). Manhattan commands the highest prices across all room types, with a median entire-home listing at roughly $190/night. Budget and economy listings (under $100) make up about 50% of the market, indicating strong competition in the affordable segment.

---

### 5.3 Room Type Analysis -- What kind of spaces are offered?

In [ ]:
# Room type distribution overall and by borough
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overall room type split
room_counts = df['room_type'].value_counts()
axes[0].pie(room_counts, labels=room_counts.index, autopct='%1.1f%%',
            colors=[room_palette[r] for r in room_counts.index],
            startangle=90, textprops={'fontsize': 11})
axes[0].set_title('Overall Room Type Distribution', fontweight='bold', fontsize=13)

# Stacked bar chart by borough
room_by_borough = df.groupby(['neighbourhood_group', 'room_type']).size().unstack(fill_value=0)
room_by_borough_pct = room_by_borough.div(room_by_borough.sum(axis=1), axis=0) * 100
room_by_borough_pct = room_by_borough_pct.reindex(borough_order)

room_by_borough_pct.plot(kind='bar', stacked=True, ax=axes[1],
                          color=[room_palette[c] for c in room_by_borough_pct.columns],
                          edgecolor='white', linewidth=0.5)
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xlabel('')
axes[1].set_title('Room Type Composition by Borough', fontweight='bold', fontsize=13)
axes[1].legend(title='Room Type', bbox_to_anchor=(1.0, 1.0))
axes[1].set_xticklabels(borough_order, rotation=0)

plt.tight_layout()
plt.savefig('room_type_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

**Key Finding:** Entire homes/apartments make up about 52% of listings, while private rooms account for 46%. Shared rooms are relatively rare at around 2%. Manhattan has a higher proportion of entire-home listings compared to Brooklyn, where private rooms are more common -- reflecting different neighbourhood characters and housing stock.

---

### 5.4 Host Analysis -- Who are the hosts?

In [ ]:
# Host type distribution
host_type_counts = df['host_type'].value_counts().reset_index()
host_type_counts.columns = ['Host Type', 'Count']
host_type_counts['Percentage'] = (host_type_counts['Count'] / len(df) * 100).round(1)

print('Host Type Distribution (by listing count):')
host_type_counts

In [ ]:
# Top 15 hosts by number of listings
top_hosts = df.groupby(['host_id', 'host_name']).agg(
    listing_count=('id', 'count'),
    avg_price=('price', 'mean'),
    total_reviews=('number_of_reviews', 'sum'),
    boroughs=('neighbourhood_group', lambda x: ', '.join(x.unique()))
).reset_index().sort_values('listing_count', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_hosts['host_name'][::-1], top_hosts['listing_count'][::-1],
               color='#3498db', edgecolor='white')

for bar_item, count in zip(bars, top_hosts['listing_count'][::-1]):
    ax.text(bar_item.get_width() + 1, bar_item.get_y() + bar_item.get_height()/2,
            f'{count}', va='center', fontweight='bold', fontsize=10)

ax.set_xlabel('Number of Listings')
ax.set_title('Top 15 Hosts by Number of Listings', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('top_hosts.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nDetailed top host information:')
top_hosts[['host_name', 'listing_count', 'avg_price', 'total_reviews', 'boroughs']].round(0)

In [ ]:
# Individual vs multi-listing hosts -- market concentration
unique_hosts = df.groupby('host_id').agg(
    listings=('id', 'count'),
    host_type=('host_type', 'first')
).reset_index()

host_summary = unique_hosts.groupby('host_type').agg(
    host_count=('host_id', 'count'),
    total_listings=('listings', 'sum')
).reset_index()

host_summary['pct_hosts'] = (host_summary['host_count'] / host_summary['host_count'].sum() * 100).round(1)
host_summary['pct_listings'] = (host_summary['total_listings'] / host_summary['total_listings'].sum() * 100).round(1)

print('Market Concentration Analysis:')
print()
host_summary

In [ ]:
# Visualise the market concentration
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

host_colors = ['#27ae60', '#3498db', '#e74c3c']
host_order_list = ['Individual Host (1 listing)', 'Small Portfolio (2-5)', 'Professional Host (6+)']

# By host count
host_pct = host_summary.set_index('host_type').loc[host_order_list, 'pct_hosts']
axes[0].pie(host_pct, labels=host_order_list, autopct='%1.1f%%',
            colors=host_colors, startangle=90, textprops={'fontsize': 10})
axes[0].set_title('Distribution of Hosts', fontweight='bold', fontsize=12)

# By listing count
listing_pct = host_summary.set_index('host_type').loc[host_order_list, 'pct_listings']
axes[1].pie(listing_pct, labels=host_order_list, autopct='%1.1f%%',
            colors=host_colors, startangle=90, textprops={'fontsize': 10})
axes[1].set_title('Distribution of Listings', fontweight='bold', fontsize=12)

plt.suptitle('Market Concentration: Hosts vs Their Listing Share', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('market_concentration.png', dpi=150, bbox_inches='tight')
plt.show()

**Key Finding:** While the majority of hosts are individuals with a single listing, a smaller group of professional hosts with 6+ listings controls a disproportionate share of the market. This concentration raises important questions about the platform's impact on housing supply and its alignment with the original "sharing economy" ethos.

---

### 5.5 Reviews and Activity Patterns

In [ ]:
# Review activity over time
reviews_by_month = df.dropna(subset=['last_review']).groupby(
    df['last_review'].dt.to_period('M')
).size().reset_index()
reviews_by_month.columns = ['Month', 'Review Count']
reviews_by_month['Month'] = reviews_by_month['Month'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(reviews_by_month['Month'], reviews_by_month['Review Count'],
                alpha=0.3, color='#3498db')
ax.plot(reviews_by_month['Month'], reviews_by_month['Review Count'],
        color='#3498db', linewidth=2)
ax.set_xlabel('Month')
ax.set_ylabel('Number of Listings with Last Review in This Month')
ax.set_title('Review Activity Timeline', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.savefig('review_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation between reviews and other metrics
correlation_cols = ['price', 'minimum_nights', 'number_of_reviews',
                    'reviews_per_month', 'calculated_host_listings_count', 'availability_365']

corr_matrix = df[correlation_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=1, linecolor='white', ax=ax,
            vmin=-1, vmax=1, square=True)
ax.set_title('Correlation Matrix of Key Numerical Variables', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Availability analysis by borough
fig, ax = plt.subplots(figsize=(12, 6))

sns.boxplot(data=df, x='neighbourhood_group', y='availability_365',
            order=borough_order, palette=borough_palette, ax=ax, linewidth=1)

ax.set_xlabel('Borough', fontsize=12)
ax.set_ylabel('Days Available per Year', fontsize=12)
ax.set_title('Listing Availability (Days per Year) by Borough', fontweight='bold', fontsize=14)

# Annotate medians
for i, borough in enumerate(borough_order):
    med = df[df['neighbourhood_group'] == borough]['availability_365'].median()
    ax.text(i, med + 10, f'Med: {med:.0f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('availability_by_borough.png', dpi=150, bbox_inches='tight')
plt.show()

---

### 5.6 Text Analysis -- What do listing names reveal?

In [ ]:
# Word cloud from listing names
all_names = ' '.join(df['name'].astype(str).tolist())

# Common stop words to filter out
stop_words = {'in', 'the', 'a', 'an', 'and', 'of', 'to', 'for', 'is', 'on', 'at',
              'with', 'from', 'or', 'by', 'near', 'it', 'its', 'new', 'york', 'nyc',
              'city', 'apt', 'br', 'bedroom', 'bed', 'room', 'home'}

wc = WordCloud(
    width=1200, height=600,
    background_color='white',
    max_words=150,
    colormap='viridis',
    stopwords=stop_words,
    collocations=False
)
wc.generate(all_names)

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Most Common Words in Listing Names', fontweight='bold', fontsize=16, pad=15)

plt.tight_layout()
plt.savefig('wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()

**Key Finding:** Hosts frequently emphasise "cozy", "private", "spacious", "beautiful", and "modern" in their listing names. Location markers like "Manhattan", "Brooklyn", "Williamsburg", and "Village" are also prominent, confirming that neighbourhood identity is a strong selling point. The prevalence of words like "studio" and "entire" suggests that many listings cater to travellers seeking self-contained accommodation.

---

### 5.7 Estimated Revenue Analysis

In [ ]:
# Revenue estimates by borough
revenue_by_borough = df.groupby('neighbourhood_group')['estimated_annual_revenue'].agg(
    ['mean', 'median', 'sum']
).round(0)
revenue_by_borough.columns = ['Mean Est. Revenue', 'Median Est. Revenue', 'Total Est. Revenue']
revenue_by_borough = revenue_by_borough.sort_values('Median Est. Revenue', ascending=False)

print('Estimated Annual Revenue by Borough:')
revenue_by_borough

In [ ]:
# Revenue distribution by borough (violin plot)
# Clip at a reasonable upper bound for readability
df_rev_clipped = df[df['estimated_annual_revenue'] <= 100000]

fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(data=df_rev_clipped, x='neighbourhood_group', y='estimated_annual_revenue',
               order=borough_order, palette=borough_palette, ax=ax,
               inner='quartile', cut=0, linewidth=1)

ax.set_xlabel('Borough', fontsize=12)
ax.set_ylabel('Estimated Annual Revenue ($)', fontsize=12)
ax.set_title('Estimated Annual Revenue Distribution by Borough', fontweight='bold', fontsize=14)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('revenue_by_borough.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top 10 revenue-generating neighbourhoods
revenue_by_neighbourhood = df.groupby(['neighbourhood', 'neighbourhood_group']).agg(
    total_revenue=('estimated_annual_revenue', 'sum'),
    listing_count=('id', 'count'),
    median_price=('price', 'median')
).reset_index().sort_values('total_revenue', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
colours_mapped = [borough_palette.get(b, '#999') for b in revenue_by_neighbourhood['neighbourhood_group']]

bars = ax.barh(revenue_by_neighbourhood['neighbourhood'][::-1],
               revenue_by_neighbourhood['total_revenue'][::-1],
               color=colours_mapped[::-1], edgecolor='white')

ax.set_xlabel('Total Estimated Annual Revenue ($)')
ax.set_title('Top 10 Revenue-Generating Neighbourhoods', fontweight='bold', fontsize=14)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f'${x/1e6:.1f}M'))

legend_elements = [Patch(facecolor=borough_palette[b], label=b) for b in borough_order]
ax.legend(handles=legend_elements, title='Borough', loc='lower right')

plt.tight_layout()
plt.savefig('top_revenue_neighbourhoods.png', dpi=150, bbox_inches='tight')
plt.show()

---

### 5.8 Minimum Nights Analysis

In [ ]:
# Minimum nights distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram (clipped at 30 for readability)
df_min_nights = df[df['minimum_nights'] <= 30]
axes[0].hist(df_min_nights['minimum_nights'], bins=30, color='#9b59b6',
             edgecolor='white', linewidth=0.5, alpha=0.85)
axes[0].axvline(x=30, color='#e74c3c', linestyle='--', linewidth=2,
                label='30-day threshold')
axes[0].set_xlabel('Minimum Nights')
axes[0].set_ylabel('Number of Listings')
axes[0].set_title('Minimum Night Requirement Distribution', fontweight='bold', fontsize=13)
axes[0].legend()

# By borough
min_night_borough = df.groupby('neighbourhood_group')['minimum_nights'].median().reindex(borough_order)
bars = axes[1].bar(borough_order, min_night_borough,
                   color=[borough_palette[b] for b in borough_order], edgecolor='white')
axes[1].set_ylabel('Median Minimum Nights')
axes[1].set_title('Median Minimum Night Requirement by Borough', fontweight='bold', fontsize=13)

for bar_item, val in zip(bars, min_night_borough):
    axes[1].text(bar_item.get_x() + bar_item.get_width()/2, bar_item.get_height() + 0.1,
                 f'{val:.0f}', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('minimum_nights.png', dpi=150, bbox_inches='tight')
plt.show()

# Proportion of listings requiring 30+ nights
long_term_pct = (df['minimum_nights'] >= 30).mean() * 100
print(f'Listings requiring 30+ nights (potential long-term rentals): {long_term_pct:.1f}%')

---

## 6. Advanced Interactive Visualisations

The static plots above tell a compelling story, but interactive charts allow stakeholders to explore the data themselves. Below we create several Plotly-based visualisations.

In [ ]:
# Interactive: Price vs Reviews scatterplot coloured by borough
sample_df = df.sample(n=5000, random_state=42)  # sample for performance

fig = px.scatter(
    sample_df[sample_df['price'] <= 1000],
    x='number_of_reviews',
    y='price',
    color='neighbourhood_group',
    color_discrete_map=borough_palette,
    size='availability_365',
    size_max=12,
    opacity=0.5,
    hover_name='name',
    hover_data={'price': ':$,.0f', 'room_type': True, 'neighbourhood': True},
    title='Price vs Number of Reviews (Sample of 5,000 Listings)',
    labels={'number_of_reviews': 'Number of Reviews', 'price': 'Price ($)',
            'neighbourhood_group': 'Borough', 'availability_365': 'Availability (days)'},
    width=900, height=550
)

fig.update_layout(title_font_size=15)
fig.show()

In [ ]:
# Interactive: Sunburst chart -- Borough > Room Type > Price Category
sunburst_data = df.groupby(['neighbourhood_group', 'room_type', 'price_category']).size().reset_index(name='count')

fig = px.sunburst(
    sunburst_data,
    path=['neighbourhood_group', 'room_type', 'price_category'],
    values='count',
    title='Hierarchical View: Borough - Room Type - Price Category',
    color='neighbourhood_group',
    color_discrete_map=borough_palette,
    width=800, height=700
)

fig.update_layout(title_font_size=15)
fig.show()

In [ ]:
# Interactive: Treemap of listings by neighbourhood
treemap_data = df.groupby(['neighbourhood_group', 'neighbourhood']).agg(
    count=('id', 'count'),
    median_price=('price', 'median')
).reset_index()

fig = px.treemap(
    treemap_data,
    path=['neighbourhood_group', 'neighbourhood'],
    values='count',
    color='median_price',
    color_continuous_scale='YlOrRd',
    title='NYC Airbnb Market: Neighbourhood Size and Median Price',
    labels={'count': 'Listings', 'median_price': 'Median Price ($)'},
    width=900, height=600
)

fig.update_layout(title_font_size=15)
fig.show()

---

## 7. Key Findings Summary

| Theme | Key Insight |
|-------|-------------|
| **Geography** | Manhattan and Brooklyn dominate with ~85% of all listings. Williamsburg, Bedford-Stuyvesant, and Harlem are the top neighbourhoods. |
| **Pricing** | Median price is $106/night. Manhattan commands the highest prices. Budget and economy segments represent ~50% of listings. |
| **Room Types** | Entire homes (52%) and private rooms (46%) dominate. Shared rooms are rare (~2%). |
| **Host Landscape** | A small number of professional hosts with many listings control a disproportionate share of the market. |
| **Reviews** | Strong growth in review activity through 2019. Higher-priced listings tend to have fewer reviews. |
| **Revenue** | Manhattan neighbourhoods generate the highest estimated revenue despite higher competition. |
| **Minimum Nights** | Most listings have a 1-3 night minimum. About 4-5% require 30+ nights, indicating potential long-term rental conversions. |

---

## 8. Recommendations

**For New Hosts:**
- Focus on the budget-to-mid-range price segment ($50-200) where demand is highest.
- Listings in Brooklyn neighbourhoods like Williamsburg and Bedford-Stuyvesant offer a good balance of demand and lower competition compared to Manhattan.
- Emphasise qualities like "cozy", "private", and "spacious" in listing names -- these resonate with guests.

**For Existing Hosts:**
- Reduce minimum night requirements to attract more bookings. Most successful listings have a 1-2 night minimum.
- Increase availability to improve visibility in search rankings.
- Focus on generating reviews; listings with more reviews tend to maintain higher booking rates.

**For Policy Makers:**
- Monitor the concentration of listings among professional hosts, particularly in high-demand areas.
- The prevalence of entire-home listings in Manhattan raises questions about housing stock impact.
- Listings with very high minimum nights (30+) may be circumventing short-term rental regulations.

**For Airbnb as a Platform:**
- The outer boroughs (Bronx, Staten Island) represent underserved markets with growth potential.
- Consider tools to help individual hosts compete with professional operators.
- Review activity growth suggests strong platform health in the NYC market.

In [ ]:
# Save the cleaned dataset for the interactive dashboard
df.to_csv('cleaned_airbnb_nyc.csv', index=False)
print(f'Cleaned dataset saved: {df.shape[0]:,} rows x {df.shape[1]} columns')
print('File: cleaned_airbnb_nyc.csv')